# ♻️ Waste VLM Dataset – Colab/Kaggle Setup Notebook

This notebook handles:
1. **Environment setup** – installs all dependencies
2. **Dataset download** – clones TACO, downloads ZeroWaste
3. **Data preparation** – runs `01_prepare_data.py` to merge & split
4. **Caption generation** – runs BLIP-2 and Qwen2.5-VL (GPU required)
5. **Evaluation** – computes CLIPScore, BLEU, CIDEr

### 🚀 Quick Start
1. Open in [Google Colab](https://colab.research.google.com) or [Kaggle](https://kaggle.com)
2. Set **Runtime → GPU** (T4 or better)
3. Run all cells in order

---

## 0. Check GPU & Mount Drive

In [ ]:
import os, sys, subprocess

# Check GPU
!nvidia-smi

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None

print(f"\n{'='*50}")
print(f"  Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"{'='*50}")

# Mount Google Drive (Colab only) – optional for persistent storage
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/waste-vlm-dataset'
elif IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/waste-vlm-dataset'
else:
    PROJECT_ROOT = os.getcwd()

print(f"  Project root: {PROJECT_ROOT}")

## 1. Clone Project & Install Dependencies

In [ ]:
%%bash -s "$PROJECT_ROOT"
PROJECT_ROOT=$1

# Clone the project repo (replace with your GitHub URL)
if [ ! -d "$PROJECT_ROOT" ]; then
    echo "📦 Cloning project repository..."
    # Uncomment and replace with your repo URL:
    # git clone https://github.com/YOUR_USERNAME/waste-vlm-dataset.git $PROJECT_ROOT
    
    # If no repo yet, create the directory structure
    mkdir -p $PROJECT_ROOT/{data/{raw,processed/{train,val,test},annotations},scripts,app,config}
    echo "✓ Created project structure"
else
    echo "✓ Project directory already exists"
fi

ls -la $PROJECT_ROOT

In [ ]:
# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 datasets>=2.18.0
!pip install -q open-clip-torch>=2.24.0
!pip install -q Pillow>=10.0.0 scikit-learn>=1.3.0 tqdm pandas
!pip install -q pycocoevalcap nltk orjson
!pip install -q qwen-vl-utils  # Qwen2.5-VL image processing utilities

# Download NLTK data for evaluation metrics
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("\n✅ All dependencies installed!")

In [ ]:
# Verify installations
import torch
import transformers
import open_clip
from PIL import Image

print(f"PyTorch:       {torch.__version__}")
print(f"CUDA avail:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Transformers:  {transformers.__version__}")
print(f"OpenCLIP:      {open_clip.__version__}")

## 2. Download Datasets

In [ ]:
%%bash -s "$PROJECT_ROOT"
PROJECT_ROOT=$1
cd $PROJECT_ROOT/data/raw

# ──────────────────────────────────────
# TACO Dataset
# ──────────────────────────────────────
if [ ! -d "TACO" ]; then
    echo "📥 Cloning TACO dataset repository..."
    git clone https://github.com/pedropro/TACO.git
    cd TACO
    pip install -q -r requirements.txt 2>/dev/null
    echo "📥 Downloading TACO images (this may take 10-20 minutes)..."
    python3 download.py
    cd ..
    echo "✓ TACO download complete"
else
    echo "✓ TACO already downloaded"
fi

echo ""
echo "TACO contents:"
ls -la TACO/ 2>/dev/null || echo "  (not found)"

In [ ]:
%%bash -s "$PROJECT_ROOT"
PROJECT_ROOT=$1
cd $PROJECT_ROOT/data/raw

# ──────────────────────────────────────
# ZeroWaste Dataset
# ──────────────────────────────────────
if [ ! -d "zerowaste" ]; then
    echo "📥 Downloading ZeroWaste dataset..."
    mkdir -p zerowaste
    
    # ZeroWaste-f (fully supervised subset)
    # Option A: Download from the official source
    # The dataset is hosted on the project page. Check:
    # https://github.com/dbash/zerowaste for the latest download links.
    
    # Option B: If you have the download link:
    # wget -q "DOWNLOAD_URL_HERE" -O zerowaste.zip
    # unzip -q zerowaste.zip -d zerowaste/
    # rm zerowaste.zip
    
    echo "⚠️  ZeroWaste requires manual download."
    echo "    Visit: https://github.com/dbash/zerowaste"
    echo "    Download zerowaste-f and extract to: $PROJECT_ROOT/data/raw/zerowaste/"
else
    echo "✓ ZeroWaste already present"
fi

echo ""
echo "ZeroWaste contents:"
ls -la zerowaste/ 2>/dev/null || echo "  (not found)"

In [ ]:
# Quick peek at TACO annotations structure
import json
from pathlib import Path

taco_ann_path = Path(PROJECT_ROOT) / 'data' / 'raw' / 'TACO' / 'data' / 'annotations.json'
if taco_ann_path.exists():
    with open(taco_ann_path) as f:
        taco = json.load(f)
    print(f"TACO Dataset Summary:")
    print(f"  Images:      {len(taco.get('images', []))}")
    print(f"  Annotations: {len(taco.get('annotations', []))}")
    print(f"  Categories:  {len(taco.get('categories', []))}")
    print(f"\n  Category names:")
    for cat in taco.get('categories', [])[:15]:
        print(f"    {cat['id']:3d}: {cat['name']}")
    if len(taco.get('categories', [])) > 15:
        print(f"    ... and {len(taco['categories']) - 15} more")
else:
    print(f"⚠️  TACO annotations not found at {taco_ann_path}")
    print("   Run the TACO download cell above first.")

## 3. Upload Project Scripts

If you cloned the repo, scripts are already in place. Otherwise, this cell writes them.

In [ ]:
import os, json
from pathlib import Path

config_dir = Path(PROJECT_ROOT) / 'config'
config_dir.mkdir(parents=True, exist_ok=True)

taxonomy_path = config_dir / 'taxonomy.json'

if not taxonomy_path.exists():
    # Write the taxonomy config
    taxonomy = {
        "version": "1.0",
        "object_categories": [
            "plastic_bottle", "glass_bottle", "metal_can", "drink_carton",
            "food_container", "plastic_bag_wrapper", "cup_lid", "cardboard_box",
            "paper_tissue", "styrofoam", "cigarette_butt", "other_litter"
        ],
        "materials": [
            "plastic_rigid", "plastic_film", "metal_aluminum", "metal_steel",
            "glass", "paper_cardboard", "composite_multilayer"
        ],
        "taco_category_map": {
            "Clear plastic bottle": {"category": "plastic_bottle", "material": "plastic_rigid"},
            "Other plastic bottle": {"category": "plastic_bottle", "material": "plastic_rigid"},
            "Plastic bottle cap": {"category": "cup_lid", "material": "plastic_rigid"},
            "Metal bottle cap": {"category": "cup_lid", "material": "metal_aluminum"},
            "Glass bottle": {"category": "glass_bottle", "material": "glass"},
            "Glass jar": {"category": "glass_bottle", "material": "glass"},
            "Broken glass": {"category": "glass_bottle", "material": "glass"},
            "Drink can": {"category": "metal_can", "material": "metal_aluminum"},
            "Food can": {"category": "metal_can", "material": "metal_steel"},
            "Aerosol": {"category": "metal_can", "material": "metal_aluminum"},
            "Drink carton": {"category": "drink_carton", "material": "composite_multilayer"},
            "Corrugated carton": {"category": "cardboard_box", "material": "paper_cardboard"},
            "Pizza box": {"category": "cardboard_box", "material": "paper_cardboard"},
            "Garbage bag": {"category": "plastic_bag_wrapper", "material": "plastic_film"},
            "Single-use carrier bag": {"category": "plastic_bag_wrapper", "material": "plastic_film"},
            "Crisp packet": {"category": "plastic_bag_wrapper", "material": "composite_multilayer"},
            "Disposable food container": {"category": "food_container", "material": "plastic_rigid"},
            "Foam food container": {"category": "food_container", "material": "plastic_rigid"},
            "Disposable plastic cup": {"category": "cup_lid", "material": "plastic_rigid"},
            "Paper cup": {"category": "cup_lid", "material": "paper_cardboard"},
            "Normal paper": {"category": "paper_tissue", "material": "paper_cardboard"},
            "Tissues": {"category": "paper_tissue", "material": "paper_cardboard"},
            "Cigarette": {"category": "cigarette_butt", "material": "composite_multilayer"},
            "Unlabeled litter": {"category": "other_litter", "material": "composite_multilayer"},
        },
        "zerowaste_category_map": {
            "rigid_plastic": {"category": "food_container", "material": "plastic_rigid"},
            "cardboard": {"category": "cardboard_box", "material": "paper_cardboard"},
            "metal": {"category": "metal_can", "material": "metal_aluminum"},
            "soft_plastic": {"category": "plastic_bag_wrapper", "material": "plastic_film"}
        }
    }
    with open(taxonomy_path, 'w') as f:
        json.dump(taxonomy, f, indent=2)
    print(f"✓ Wrote taxonomy config to {taxonomy_path}")
else:
    print(f"✓ Taxonomy config already exists at {taxonomy_path}")

## 4. Run Week 1 – Data Preparation

Merge TACO & ZeroWaste annotations, apply taxonomy mapping, split 70/15/15.

In [ ]:
os.chdir(PROJECT_ROOT)

# If the script exists locally, run it. Otherwise, inline the logic.
script_path = Path(PROJECT_ROOT) / 'scripts' / '01_prepare_data.py'

if script_path.exists():
    !python scripts/01_prepare_data.py \
        --taco_dir data/raw/TACO \
        --zw_dir data/raw/zerowaste \
        --output data/annotations/week1_base.json \
        --img_out data/processed \
        --seed 42
else:
    print("⚠️  Script not found. Upload your project scripts or clone the repo.")
    print(f"   Expected at: {script_path}")

In [ ]:
# Verify the output
ann_path = Path(PROJECT_ROOT) / 'data' / 'annotations' / 'week1_base.json'
if ann_path.exists():
    with open(ann_path) as f:
        data = json.load(f)
    print(f"✅ Week 1 base annotations:")
    print(f"   Images:  {data['info']['total_images']}")
    print(f"   Objects: {data['info']['total_objects']}")
    
    # Show first record
    if data['annotations']:
        print(f"\n   Sample record:")
        sample = data['annotations'][0]
        print(json.dumps({k: v for k, v in sample.items() if k != 'objects'}, indent=2))
        if sample['objects']:
            print(f"\n   First object:")
            print(json.dumps(sample['objects'][0], indent=2))
else:
    print("⚠️  Annotations file not created. Check the preparation step above.")

## 5. Week 2 – Caption Generation (GPU Required)

Generate captions using **BLIP-2** and **Qwen2.5-VL**.

> ⚡ This cell requires a **GPU runtime**. On Colab: Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Configuration ──
MODELS_TO_RUN = ['blip2', 'qwen']   # Options: 'blip2', 'qwen'
MAX_IMAGES = 50                       # Set to 0 for all images
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
print(f"Models: {MODELS_TO_RUN}")
print(f"Max images: {MAX_IMAGES if MAX_IMAGES > 0 else 'ALL'}")

In [ ]:
os.chdir(PROJECT_ROOT)

script_path = Path(PROJECT_ROOT) / 'scripts' / '02_generate_captions.py'

if script_path.exists():
    limit_flag = f"--limit {MAX_IMAGES}" if MAX_IMAGES > 0 else ""
    models_flag = " ".join(MODELS_TO_RUN)
    
    !python scripts/02_generate_captions.py \
        --annotations data/annotations/week1_base.json \
        --output data/annotations/week2_generated.json \
        --img_root data/processed \
        --models {models_flag} \
        --device {DEVICE} \
        {limit_flag}
else:
    print("⚠️  Script not found. Upload scripts/02_generate_captions.py")

## 6. Preview Generated Captions

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import textwrap

gen_path = Path(PROJECT_ROOT) / 'data' / 'annotations' / 'week2_generated.json'
if not gen_path.exists():
    gen_path = Path(PROJECT_ROOT) / 'data' / 'annotations' / 'week1_base.json'

if gen_path.exists():
    with open(gen_path) as f:
        gen_data = json.load(f)
    
    # Show up to 6 sample images with captions
    samples = [a for a in gen_data['annotations'] if a.get('scene_caption_blip2') or a.get('scene_caption_qwen')][:6]
    if not samples:
        samples = gen_data['annotations'][:6]
    
    n = len(samples)
    if n > 0:
        fig, axes = plt.subplots(2, min(n, 3), figsize=(18, 12))
        if n == 1:
            axes = [[axes]]
        elif n <= 3:
            axes = [axes] if n > 1 else [[axes]]
        
        for idx, sample in enumerate(samples[:6]):
            row = idx // 3
            col = idx % 3
            
            img_path = Path(PROJECT_ROOT) / sample['file_name']
            if img_path.exists():
                img = Image.open(img_path)
                ax = axes[row][col] if isinstance(axes[0], list) else axes[col]
                ax.imshow(img)
                
                blip2 = sample.get('scene_caption_blip2', 'N/A')[:100]
                qwen = sample.get('scene_caption_qwen', 'N/A')[:100]
                title = f"{sample['image_id']}\n"
                title += f"BLIP-2: {textwrap.fill(blip2, 40)}\n"
                title += f"Qwen: {textwrap.fill(qwen, 40)}"
                ax.set_title(title, fontsize=8, ha='left', x=0)
                ax.axis('off')
        
        plt.tight_layout()
        plt.savefig(Path(PROJECT_ROOT) / 'caption_preview.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✓ Preview saved to caption_preview.png")
    else:
        print("No samples to preview.")
else:
    print("⚠️  No annotation files found. Run data preparation first.")

## 7. Week 4 – Evaluation

In [ ]:
os.chdir(PROJECT_ROOT)

eval_input = 'data/annotations/week2_generated.json'
if not Path(PROJECT_ROOT, eval_input).exists():
    eval_input = 'data/annotations/week1_base.json'

script_path = Path(PROJECT_ROOT) / 'scripts' / '03_evaluate.py'
if script_path.exists() and Path(PROJECT_ROOT, eval_input).exists():
    !python scripts/03_evaluate.py \
        --annotations {eval_input} \
        --output data/annotations/week4_evaluated.json \
        --img_root data/processed \
        --device {DEVICE}
else:
    print("⚠️  Missing scripts or annotations. Run previous steps first.")

## 8. Download Results

Download the final annotation files to your local machine.

In [ ]:
# List all generated files
ann_dir = Path(PROJECT_ROOT) / 'data' / 'annotations'
if ann_dir.exists():
    print("📁 Annotation files:")
    for f in sorted(ann_dir.glob('*.json')):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"   {f.name:<35s}  {size_mb:.2f} MB")

# Download in Colab
if IN_COLAB:
    from google.colab import files
    for f in ann_dir.glob('*.json'):
        files.download(str(f))
        print(f"   ⬇ Downloaded {f.name}")

# Copy to Drive for persistence (Colab)
if IN_COLAB:
    drive_dest = '/content/drive/MyDrive/waste-vlm-dataset-output'
    os.makedirs(drive_dest, exist_ok=True)
    !cp -r {PROJECT_ROOT}/data/annotations/* {drive_dest}/
    print(f"\n✓ Copied annotations to Google Drive: {drive_dest}")

## 9. Copy to Local Project (Optional)

If you're working locally, sync the Colab outputs back to your project.

In [ ]:
# Create a zip of all outputs for easy download
import shutil

zip_path = Path(PROJECT_ROOT) / 'waste_vlm_outputs'
if ann_dir.exists():
    shutil.make_archive(str(zip_path), 'zip', str(ann_dir))
    print(f"✓ Created {zip_path}.zip")
    
    if IN_COLAB:
        files.download(f"{zip_path}.zip")

---
## ✅ Done!

**Next steps:**
1. Download the annotation JSON files
2. Copy them to your local `data/annotations/` folder
3. Run the Streamlit app: `streamlit run app/streamlit_app.py`
4. Review, edit, and approve captions
5. Export the final dataset

**Week 3 (Manual Verification)** is done via the Streamlit app.

---
*Waste VLM Dataset Project – Vision-Language Caption Generation for Waste Images*